In [8]:
from datasets import load_dataset, concatenate_datasets

ds_2024 = load_dataset("Reubencf/2024_events")
ds_2025 = load_dataset("Reubencf/2025_events")

combined_events = concatenate_datasets([
    ds_2024["train"],
    ds_2025["train"]
])

# normalize section
events = combined_events.map(
    lambda x: {"section": x["section"].lower()}
)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time
import json

llm = ChatOllama(model="llama3.2:3b")

ei = 0
errs = 0

start_time = time.time()

print("Generating Q&As for training and validation:")


def format_time(seconds):
    seconds = int(seconds)
    mins, secs = divmod(seconds, 60)
    hours, mins = divmod(mins, 60)
    
    if hours:
        return f"{hours}h {mins:02d}m {secs:02d}s"
    else:
        return f"{mins:02d}m {secs:02d}s"

def validate_qa_json(data):
    if not isinstance(data, dict):
        return False
    
    if "qa" not in data:
        return False
    
    qa_list = data["qa"]
    
    # Must be list of length 3
    if not isinstance(qa_list, list) or len(qa_list) != 3:
        return False
    
    valid_types = {"fact", "causal", "impact"}
    
    for item in qa_list:
        if not isinstance(item, dict):
            return False
        
        if not all(k in item for k in ["question", "answer", "type"]):
            return False
        
        if item["type"] not in valid_types:
            return False
    
    return True
    
data = None
qa_dataset = []

for event in events:
    date_str = f"{event['day']} {event['month']} {event['year']}"
    
    prompt = ChatPromptTemplate.from_template(
        """
            You are an expert dataset generator.
            
            Given this news article:
            
            {date_str}

            {content}

            {response}


            --- End of article ---
            
            Generate 3 high-quality QA pairs:
            
            1. Factual question (what/who/when/where)
            2. Causal question (why/how)
            3. Impact question (what were the consequences)
            
            Rules:
            - Answers must be concise but complete (3–6 sentences)
            - Must strictly use information from the context
            - Use the date of the article in the questions and answers where applicable.
            - No hallucination
            - Only output the JSON and nothing else like you are giving an API response
            - Do not return any other JSON format
            - Length of questions and answers individually must ideally be more than 20 characters
            
            Return STRICT JSON:
            {{
              "qa": [
                {{"question": "...", "answer": "...", "type": "fact"}},
                {{"question": "...", "answer": "...", "type": "causal"}},
                {{"question": "...", "answer": "...", "type": "impact"}}
              ]
            }}

            Only return JSON in the given schema and no other.
        """
    )
    
    data = None
    
    while data is None:
        
        chain = prompt | llm | StrOutputParser()
    
        json_output = chain.invoke({
            "date_str": date_str,
            "content": event["content"],
            "response": event["response"]
        })
        
        clean_json = json_output[json_output.find("{"):json_output.rfind("}") + 1]

        # print(clean_json)
        if clean_json:
            try:
                data = json.loads(clean_json)

                if validate_qa_json(data):
                    ei += 1

                    for qa in data["qa"]:
                        qa_dataset.append({
                            "input": qa["question"],
                            "output": qa["answer"],
                            "type": qa["type"],
                            "date": date_str,
                            "eidx": ei
                        })

                else:
                    errs += 1
                    # print(json.dumps(data, indent=2, ensure_ascii=False)) 
                    data = None
            except Exception:
                data = None
                errs += 1
    
            elapsed = time.time() - start_time
            progress = ei / len(events)
    
            if progress == 0:
                eta = "Unknown"
            else:
                eta = format_time(elapsed/progress * (1-progress))
        else:
            errs += 1
            
        print(f"\r{ei} / {len(events)} [ {round((ei) / len(events) * 100, 2)}% ] | Errs: {errs} | ETA: {eta}      ", end="")

    # if ei > 5:
    #     break;

with open("qa_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in qa_dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

Generating Q&As for training and validation:
40 / 10712 [ 0.37% ] | Errs: 2 | ETA: 7h 45m 04s      